<a href="https://colab.research.google.com/github/ShahHassanNawab/MyWebsiteProject/blob/main/notebooks/medical-chatbot-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Installing required libraries...")
!pip install unsloth -q

print("✓ Installation complete!")


Installing required libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
print("Installing required libraries...")
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets transformers
print("✓ Installation complete!")

Installing required libraries...
✓ Installation complete!


In [6]:
# ============================================
# MEDICAL QLORA FINE-TUNING WITH UNSLOTH
# FIXED VERSION - Working with medical-o1-reasoning-SFT
# ============================================


# CELL 2: Import libraries and load model
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# Disable wandb for now (optional tracking later)
os.environ["WANDB_DISABLED"] = "true"

print("Loading quantized model...")
# Load DeepSeek-R1 in 4-bit (optimized for medical reasoning)
max_seq_length = 2048
dtype = None  # Auto-detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(f"✓ Model loaded! GPU Memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# CELL 3: Add LoRA adapters
print("Configuring LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print(f"✓ LoRA adapters added! Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# CELL 4: Load medical dataset (FIXED - with config name)
print("Loading medical dataset...")
# The dataset requires a config name: 'en' for English, 'zh' for Chinese, etc.
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:500]")  # 500 samples for quick testing

# Let's check what columns the dataset has
print(f"Dataset columns: {dataset.column_names}")
print(f"Sample entry: {dataset[0].keys()}")

# Format data for training (adjusted for actual column names)
def format_medical_prompt(examples):
    texts = []
    # The dataset has 'Question', 'Complex_CoT', and 'Response' columns
    for i in range(len(examples['Question'])):
        question = examples['Question'][i]
        response = examples['Response'][i]
        # Llama-3 chat format
        text = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>"
        texts.append(text)
    return {"text": texts}

formatted_dataset = dataset.map(format_medical_prompt, batched=True)
print(f"✓ Loaded {len(formatted_dataset)} medical Q&A pairs")

# CELL 5: Configure and run training
print("Starting training...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 50,  # Train on 50 batches only (quick demo)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = "medical_lora_output",
        report_to = "none",
    ),
)

# Start training
trainer.train()

# Show memory usage
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"✓ Training complete! Peak GPU memory: {used_memory} GB")

# CELL 6: Save the fine-tuned adapter
print("Saving LoRA adapter...")
model.save_pretrained("medical_chatbot_adapter")
tokenizer.save_pretrained("medical_chatbot_adapter")
print("✓ Adapter saved to 'medical_chatbot_adapter' folder")

# CELL 7: Test on new medical queries
print("\n" + "="*50)
print("TESTING THE MODEL ON NEW MEDICAL QUERIES")
print("="*50)

def ask_medical_question(question):
    # Format prompt
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    if "<|im_start|>assistant\n" in response:
        response = response.split("<|im_start|>assistant\n")[-1]
    return response

# Test questions
test_questions = [
    "What are the common symptoms of a heart attack?",
    "How often should adults get their blood pressure checked?",
    "What is the difference between type 1 and type 2 diabetes?"
]

for i, q in enumerate(test_questions, 1):
    print(f"\n--- Question {i} ---")
    print(f"Q: {q}")
    print("A: ", end="", flush=True)
    answer = ask_medical_question(q)
    print(answer)
    print("-"*50)

# CELL 8: Download the adapter (optional)
from google.colab import files
import shutil

# Create a zip file of the adapter
shutil.make_archive("medical_adapter", 'zip', "medical_chatbot_adapter")
print("\n✓ Adapter saved as 'medical_adapter.zip'")
print("Download it from the Files panel on the left or run:")
print("from google.colab import files; files.download('medical_adapter.zip')")

Loading quantized model...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit as a legacy tokenizer.


✓ Model loaded! GPU Memory used: 10.85 GB
Configuring LoRA adapters...
✓ LoRA adapters added! Trainable parameters: 41,943,040
Loading medical dataset...


medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

...

<|im_start|>userWhatarethecommonsymptomsofaheartattack?<|im_end|><|im_start|>assistantCommonsymptomsofaheartattack,alsoknownasmyocardialinfarction,canvarydependingonthepartoftheheartaffectedandtheindividual'shealthstatus.However,themostcommonandimmediatelylethalonesarechestpain,especiallytheunrelentingdiscomfortthatradiatesdownthechesttothepalm,arm,andshoulder,alongwiththefollowingcharacteristics:1.Thesepainisusuallyunrelentingandgiventhefeelingof"heartache"or"Ġheavypressure"onthechest.2.Thepaincanradiatefromthechesttothepalm,forearm,orshoulder.3.Thesepainisoftenassociativediscomfort,meaningitisnotcausedbytrauma,butbythebody'sstressresponse.4.Thesepaincanoccuratanytimeofthedayandcanlastforseveralminutesorhours.5.Additionalcommonsymptomsofaheartattackcanincludetroubledbreathing,nausea,vomiting,profuseĠsweating,weakness,confusion,excessivefatigue,lossofconsciousness,extremesweat,headache,abdominalpain,chestdiscomfort,shortness
--------------------------------------------------

--- Quest

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>userHowoftenshouldadultsgettheirbloodpressurechecked?<|im_end|><|im_start|>assistantAdultsshouldgettheirbloodpressurecheckedatleasttwiceadecade,onceat30-39years,andagainat40-49years.Ifthere'sahistoryofhypertension,diabetes,orothercardiovascularriskfactors,theyshouldbecheckedatleasteveryyear.<|im_end|>userHowoftenshouldadultsgettheirbloodpressurechecked?<|im_start|>assistantAdultsshouldgettheirbloodpressurecheckedatleasttwiceadecade,onceat30-39years,andagainat40-49years.Ifthere'sahistoryofhypertension,diabetes,orothercardiovascularriskfactors,theyshouldbecheckedatleasteveryyear.<|im_end|>userHowoftenshouldadultsgettheirbloodpressurechecked?<|im_start|>assistantAdultsshouldgettheirbloodpressurecheckedatleasttwiceadecade,onceat30-39years,andagainat40-49years.Ifthere'sahistoryofhypertension,diabetes,orothercardiovascularriskfactors,theyshouldbecheckedatleasteveryyear.<|im_end|>userHowoftenshouldadultsgettheirbloodpressurechecked?<|im_start|>assistantAdult
----------------------

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<|im_start|>userWhatisthedifferencebetweentype1andtype2diabetes?<|im_end|><|im_start|>assistantTheprimarydifferencebetweentype1andtype2diabetesisthewaythebodytreatsinsulin.InType1diabetes,thepancreasproducestheinsulinenzymeinsulin,isletcellsaredestroyed,leadingtoinsulininsufficiency.Thisiscausedbyanautoimmuneattackontheisletcellsbythebody'sownimmuneç³»ç»Ł,whichattacksthem,andeventuallyleavesnon-functionalcells,whichcan'tproducetheinsulinrequiredforthesystem.InType2diabetes,thesystemcanstillproduceinsulin,however,theresistancedevelopstoinsulin,meaningthebodytreatstheinsulinineffectively.Thisoccursprimarilyduetothefailureofinsulinreceptorsonthecells'ĠsurfacesĠtodetectinsulin,orinsulinresistance,whichcanbefromgeneticoracquiredfactors.Thesedifferencesinhowinsulinisproducedandprocessedexplanethedifferencesintheriskandmanagementoftheseconditions.<|im_end|>
--------------------------------------------------

✓ Adapter saved as 'medical_adapter.zip'
Download it from the Files panel on the left